# Fitbit Fitness Tracker Data - Project Setup

This notebook records the environment, input inventory, checksums,
parsing policy, loader plan, and access assumptions shared by
Simple and Advanced EDA.

## Step 0: Confirm setup and permitted inputs

### Notebook setup

Imports and project paths used by every setup check.
Stop early if the kernel is not the project's `.venv` (details in 0.2).

In [1]:
import hashlib
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

# Find the repository root, even when the notebook runs inside EDA/.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        folder
        for folder in (cwd, *cwd.parents)
        if (folder / ".git").exists()
        and (folder / "requirements.txt").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root containing .git and requirements.txt"
    )

# Stop if the notebook uses the wrong environment.
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project's Fitbit kernel before continuing.")

DATA_ROOT = PROJECT_ROOT / "Dataset"
MANIFEST_PATH = PROJECT_ROOT / "EDA" / "input_manifest.csv"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"
KERNEL_SPEC_PATH = (
    PROJECT_ROOT / ".venv" / "share" / "jupyter" / "kernels"
    / "fitbit-fitness-tracker" / "kernel.json"
)

# Import pandas after checking the environment.
import pandas as pd

print("Working directory:", cwd)
print("Project root     :", PROJECT_ROOT)
print("Project environment: OK")

Working directory: D:\Projects\AI and ML\FitBit Fitness Tracker Data\EDA
Project root     : D:\Projects\AI and ML\FitBit Fitness Tracker Data
Project environment: OK


### 0.1 Project root and existing setup

Locate the repository root independently of the notebook's working directory.

Check the existing Git repository, virtual environment, CI workflow,
and registered Fitbit kernel. Reuse this setup without reinstalling
packages or resetting project files.

Record the current branch, baseline commit, and uncommitted changes.
The baseline commit may not include the notebook changes currently being made.

In [2]:
def run_git(*arguments):
    """Run a read-only Git command in this project."""
    result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), *arguments],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()


# These commands inspect Git; they do not change the repository.
git_branch = run_git("branch", "--show-current")
git_commit = run_git("rev-parse", "HEAD")
git_changes = run_git("status", "--short")

setup_paths = {
    "Git metadata": PROJECT_ROOT / ".git",
    "Virtual environment": PROJECT_ROOT / ".venv",
    "CI workflow": PROJECT_ROOT / ".github" / "workflows" / "ci.yml",
    "Requirements": REQUIREMENTS_PATH,
    "Dataset folder": DATA_ROOT,
    "EDA folder": PROJECT_ROOT / "EDA",
    "Fitbit kernel specification": KERNEL_SPEC_PATH,
}

print("Branch         :", git_branch)
print("Baseline commit:", git_commit)

print("\nExisting setup:")
for label, path in setup_paths.items():
    print(f"{label:<30} exists: {path.exists()}")

print("\nUncommitted changes:")
print(git_changes or "Working tree clean")

Branch         : eda/1-simple-eda
Baseline commit: 5ce16d0366d813d058618943250a0c506ead2bb6

Existing setup:
Git metadata                   exists: True
Virtual environment            exists: True
CI workflow                    exists: True
Requirements                   exists: True
Dataset folder                 exists: True
EDA folder                     exists: True
Fitbit kernel specification    exists: True

Uncommitted changes:
M EDA/Project_Setup.ipynb
 M "EDA/Simple EDA.ipynb"
 M EDA/analysis_brief.md
 M docs/data_notes.md
?? EDA/distinct_value_counts.csv


### 0.2 Environment verification and CI scope

Confirm that the notebook's Python process belongs to this project's .venv. 

Record Python, OS, and installed direct-dependency versions.
Compare package versions with requirements.txt.

CI checks: 
- Dependency installation.
- Dependency compatibility using pip check.
- Import of the selected analysis and notebook packages.

CI does not load the Fitbit dataset, execute this notebook,
or establish that EDA findings are correct.

In [3]:
print("Python version     :", sys.version.split()[0])
print("Python executable  :", sys.executable)
print("OS                 :", platform.platform())
print("Virtual environment:", sys.prefix != sys.base_prefix)
print("Project .venv      : True (checked in the setup cell)")

# Read the direct-version pins recorded in requirements.txt.
required_versions = {}

for line in REQUIREMENTS_PATH.read_text(encoding="utf-8-sig").splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "==" in line:
        package, pinned_version = line.split("==", 1)
        required_versions[package.strip()] = pinned_version.strip()

package_rows = []

for package, pinned_version in required_versions.items():
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        installed_version = "NOT INSTALLED"

    package_rows.append({
        "package": package,
        "installed": installed_version,
        "required": pinned_version,
        "matches_pin": installed_version == pinned_version,
    })

package_versions = pd.DataFrame(package_rows)

print("\nAll pins match:", bool(package_versions["matches_pin"].all()))
display(package_versions)

Python version     : 3.12.10
Python executable  : D:\Projects\AI and ML\FitBit Fitness Tracker Data\.venv\Scripts\python.exe
OS                 : Windows-11-10.0.26200-SP0
Virtual environment: True
Project .venv      : True (checked in the setup cell)

All pins match: True


,package,installed,required,matches_pin
0,pandas,3.0.6,3.0.6,True
1,numpy,2.5.4,2.5.4,True
2,matplotlib,3.11.2,3.11.2,True
3,seaborn,0.13.2,0.13.2,True
4,jupyterlab,4.6.4,4.6.4,True
5,ipykernel,7.4.0,7.4.0,True


### 0.3 Identify the local inputs

The local dataset contains two export folders, labelled P1 and P2.
Folder labels identify exports; actual observation dates remain to be verified.

Record CSV filenames, export membership, exact byte sizes and a SHA-256
checksum for every file, and save them in `EDA/input_manifest.csv`.

The checksums identify the file contents used in this analysis.
They allow future runs to detect content changes, but do not prevent
editing or establish data accuracy. Hashing reads file bytes without
changing the CSV files.

Reference pages:
- Kaggle: https://www.kaggle.com/datasets/arashnic/fitbit
- Original study deposit: https://zenodo.org/records/53894

In [4]:
PERIOD_DIRS = {
    "P1": (
        DATA_ROOT
        / "mturkfitbit_export_3.12.16-4.11.16"
        / "Fitabase Data 3.12.16-4.11.16"
    ),
    "P2": (
        DATA_ROOT
        / "mturkfitbit_export_4.12.16-5.12.16"
        / "Fitabase Data 4.12.16-5.12.16"
    ),
}

for source_folder, folder in PERIOD_DIRS.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"{source_folder} export folder not found: {folder}")


def identify_source_folder(path):
    """Return the export label (P1/P2) for a discovered file."""
    for source_folder, folder in PERIOD_DIRS.items():
        if path.parent.resolve() == folder.resolve():
            return source_folder

    raise ValueError(f"CSV outside the configured export folders: {path}")


csv_paths = sorted(DATA_ROOT.rglob("*.csv"))

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under: {DATA_ROOT}")

inventory_rows = []

for path in csv_paths:
    # Hash the file without loading the table into memory.
    with path.open("rb") as file:
        sha256 = hashlib.file_digest(file, "sha256").hexdigest()

    # source_folder is the export label; source_file is relative to Dataset/.
    inventory_rows.append({
        "source_folder": identify_source_folder(path),
        "source_file": path.relative_to(DATA_ROOT).as_posix(),
        "file_name": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256,
    })

input_inventory = pd.DataFrame(inventory_rows)
input_inventory["size_mb"] = (input_inventory["size_bytes"] / 1_000_000).round(2)

period_summary = input_inventory.groupby("source_folder").agg(
    csv_files=("source_file", "size"),
    size_bytes=("size_bytes", "sum"),
)
period_summary["size_mb"] = (period_summary["size_bytes"] / 1_000_000).round(2)

print("CSV files found:", len(input_inventory))
print("Total bytes    :", input_inventory["size_bytes"].sum())
display(period_summary)

# Metadata only, not participant records; full paths are in the manifest.
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(input_inventory.drop(columns=["source_file"]))

CSV files found: 29
Total bytes    : 616796545


,csv_files,size_bytes,size_mb
source_folder,,,
P1,11,278798829,278.8
P2,18,337997716,338.0


,source_folder,file_name,size_bytes,sha256,size_mb
0,P1,dailyActivity_merged.csv,51260,b56bf67d7eb322da48d74680aa7ea6552aa4cdc83759ca6f056e2579ff77dac7,0.05
1,P1,heartrate_seconds_merged.csv,41069585,feabb4f4dcfb4a64f1616a01a940fef4cd7c516d493d46582e00745dd37504a2,41.07
2,P1,hourlyCalories_merged.csv,872698,3fa7b2bd6fce95016253c3cbfc71eed32e7291da5bd172f97da59540aa82bd8f,0.87
3,P1,hourlyIntensities_merged.csv,971718,cf79383517d3724c3b63fb55f38c190d3a145081dc025e4bd0a696c9253c95a9,0.97
4,P1,hourlySteps_merged.csv,865300,8c54359bb8e854e22a1ada24855f6ccdc79416288ecbcdee2eee227113d7cd48,0.87
5,P1,minuteCaloriesNarrow_merged.csv,72461949,493884f65c6880daab34159f6c839c6c832bf1322aac47fab43a66804a6d2d64,72.46
6,P1,minuteIntensitiesNarrow_merged.csv,50523569,86dd7c73c6a0989be5a9d168073d071d1672e36e2fa0316ca49ef6c94fd65698,50.52
7,P1,minuteMETsNarrow_merged.csv,51970880,f5f71bb0a5c6f378dbe9fa528acdcd43a62e9f8e8583cdec3f9bb9e5a506e73d,51.97
8,P1,minuteSleep_merged.csv,9316325,6cb2bcd0ea20e782233c886cce1731e1cd3ace12fc230f7c06b7a7d13067e7a3,9.32
9,P1,minuteStepsNarrow_merged.csv,50692143,9776fc1f3ca7dd22184e6c0ab42dd05167bf268ab5e6b1b176ccefc03d67fc14,50.69


**Save the metadata manifest.** An existing manifest is never replaced
silently: the current files must match it exactly.

In [5]:
# Store exact byte sizes; rounded MB values are only for display.
manifest_table = (
    input_inventory[["source_folder", "source_file", "size_bytes", "sha256"]]
    .sort_values(["source_folder", "source_file"])
    .reset_index(drop=True)
)

if MANIFEST_PATH.exists():
    existing_manifest = pd.read_csv(
        MANIFEST_PATH,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    if list(existing_manifest.columns) != list(manifest_table.columns):
        raise ValueError(
            "Existing manifest uses a different format. Review it before replacing it."
        )

    existing_manifest = (
        existing_manifest
        .sort_values(["source_folder", "source_file"])
        .reset_index(drop=True)
    )

    if not existing_manifest.equals(manifest_table.astype("string")):
        raise ValueError(
            "Current filenames, export membership, sizes or checksums differ from the manifest."
        )

    print("Existing manifest matches the current files, sizes and checksums.")
else:
    manifest_table.to_csv(MANIFEST_PATH, index=False, encoding="utf-8", mode="x")
    print("Metadata manifest created:", MANIFEST_PATH)

Existing manifest matches the current files, sizes and checksums.


### 0.4 Parsing policy

Initial ingestion will preserve source field values as strings.

- IDs remain text identifiers.
- Empty strings and tokens such as NA, N/A, and NULL remain unchanged until their meanings are examined.
- Zero remains distinct from missing data.
- Date/time strings remain unchanged during initial ingestion.
- Start with UTF-8/BOM-compatible decoding and strict decoding errors. If decoding fails, investigate rather than silently replacing characters.
- Use comma separation provisionally and validate the file structure.
- Do not skip blank records or parser-reported malformed rows.
- Check duplicate/empty headers and record widths during loader validation.

Candidate date formats to verify:
- Date: month/day/year.
- Datetime: month/day/year followed by 12-hour time and AM/PM.

Later derived columns may use:
- Nullable integers for verified integral counts.
- Nullable floating-point values for numeric measurements.
- Explicit datetime formats after inspection.

Retain original strings alongside relevant conversions.
Do not round analytical values or assign an unknown timezone.

Parser settings are a planned policy, not evidence that every file
has already passed parsing or structural validation.

In [6]:
READ_POLICY = {
    "sep": ",",
    "encoding": "utf-8-sig",
    "encoding_errors": "strict",
    "dtype": "string",
    "na_filter": False,
    "parse_dates": False,
    "skip_blank_lines": False,
    "on_bad_lines": "error",
}

# Proposed formats only. Do not apply them until inspected and verified.
CANDIDATE_DATE_FORMATS = {
    "date": "%m/%d/%Y",
    "datetime": "%m/%d/%Y %I:%M:%S %p",
}

print("Initial CSV parsing policy:")
for setting, value in READ_POLICY.items():
    print(f"{setting:<20}: {value}")

print("\nCandidate date formats:")
for field_type, date_format in CANDIDATE_DATE_FORMATS.items():
    print(f"{field_type:<20}: {date_format}")

Initial CSV parsing policy:
sep                 : ,
encoding            : utf-8-sig
encoding_errors     : strict
dtype               : string
na_filter           : False
parse_dates         : False
skip_blank_lines    : False
on_bad_lines        : error

Candidate date formats:
date                : %m/%d/%Y
datetime            : %m/%d/%Y %I:%M:%S %p


### 0.5 Loader and row-provenance plan

Use the inventoried files as the input registry.

The loader will:
- Select a configured export and filename explicitly.
- Begin with dailyActivity_merged.csv.
- Read one file at a time.
- Offer chunked reading for large files.
- Preserve the initial string representation and parsing policy.
- Validate headers, record widths, and provenance-column collisions.
- Retain source_folder, source_file, and source_row.

Provenance definitions:
- source_folder: export label, P1 or P2.
- source_file: path relative to Dataset/.
- source_row: one-based data-record number within the source file,
  excluding the header; not necessarily a physical text-line number.

Chunked loading must continue source_row numbering across chunks.

The loader with provenance columns is not built yet: build it when a step
first needs full records with provenance (expected in Step 4).
Steps 2-3 in `Simple EDA.ipynb` read files in chunks with `read_chunks()`
and the same parsing policy, without provenance columns.
No participant records are loaded in Step 0.

In [7]:
# Build the registry from the files discovered in 0.3.
INPUT_FILES = {}

for entry in input_inventory.itertuples(index=False):
    key = (entry.source_folder, entry.file_name)

    if key in INPUT_FILES:
        raise ValueError(f"Ambiguous input registry key: {key}")

    INPUT_FILES[key] = DATA_ROOT / entry.source_file

LOADER_PLAN = {
    "first_file": "dailyActivity_merged.csv",
    "initial_exports": ["P1", "P2"],
    "default_chunk_rows": 100_000,
    "provenance_columns": ["source_folder", "source_file", "source_row"],
    "implementation_status": "Planned; build when first needed (expected in Step 4)",
}

# Verify the proposed first inputs exist without reading their records.
for source_folder in LOADER_PLAN["initial_exports"]:
    key = (source_folder, LOADER_PLAN["first_file"])

    if key not in INPUT_FILES:
        raise FileNotFoundError(f"Proposed first input not found: {key}")

    print(source_folder, INPUT_FILES[key].relative_to(DATA_ROOT).as_posix())

print("\nRegistered CSV files:", len(INPUT_FILES))
print("Loader status:", LOADER_PLAN["implementation_status"])

P1 mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/dailyActivity_merged.csv
P2 mturkfitbit_export_4.12.16-5.12.16/Fitabase Data 4.12.16-5.12.16/dailyActivity_merged.csv

Registered CSV files: 29
Loader status: Planned; build when first needed (expected in Step 4)


### 0.6 Data handling

- Keep raw CSVs unchanged in `Dataset/`.
- Keep `Dataset/` and `.venv/` excluded from Git.
- Review notebook outputs before committing; avoid unnecessary local paths.
- Original participant and log IDs may appear in record examples by your choice.
- Keep aggregates and the input manifest with the project.
- Uploading raw data is only a possible future step.

**Earlier decision (2026-10-17):** keep previews with person IDs replaced by
codes and log IDs hidden.

**Updated 2026-10-19:** original IDs may be shown in examples.
This replaces the earlier ID-hiding rule.


### Step 0 - Final review

Before committing:

- [ ] Restart the kernel and run all setup cells successfully.
- [ ] Confirm the project environment and package versions pass their checks.
- [ ] Confirm the manifest records all 29 CSVs, their sizes, and SHA-256 checksums.
- [ ] Confirm `Dataset/` and `.venv/` are ignored and contain no tracked files.
- [ ] Review notebook outputs before sharing.

Next: the Step 1 brief in [analysis_brief.md](analysis_brief.md), then
Steps 2+ in [Simple EDA.ipynb](<Simple EDA.ipynb>).